## Model evaluation: self/other and valence

This notebook compares the classifications of different GPT and Claude models against manually annotated manifesto sentences. It reports accuracy, F1 scores, Cohen’s kappa, disagreement matrices, processing time and estimated API costs.

In [39]:
import pandas as pd
import os
import numpy as np
import re
from pathlib import Path

In [36]:
from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/West_Narratives"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
#import files
INPUT_FILE = f"{PROJECT_DIR}/sample_coders_200826_agreement.xlsx - Sheet1.csv"

df_coded = pd.read_csv(INPUT_FILE)

print(df_coded.shape)
df_coded.head()


(200, 20)


,docname,from,to,pre,keyword,post,lenght,text,referent_category,self/other_SP,valence_SP,self/other_C1,valence_C1,self/other_C2,valence_C2,self/other_C3,valence_C3,agreement_self_other,agreement_valence,comments
0,97522_201407,1647,1647,National Parliaments must be more closely invo...,"NSi understands that national parliaments, tog...","In order to shape the future, it is essential ...",16,National Parliaments must be more closely invo...,"EU, NATO and European structures",2,2,2,1,2,1,2,2,2,2,NaN
1,88440_201210,373,373,We will ensure Lithuania's security through ac...,We will effectively participate in the resolut...,Ensuring that Lithuanian representatives are a...,25,We will ensure Lithuania's security through ac...,"EU, NATO and European structures",1,1,1,1,-1,1,1,2,1,1,NaN
2,97330_200410,1750,1750,5.,Air transport modernisation of air traffic con...,6.,87,5. Air transport modernisation of air traffic ...,"EU, NATO and European structures",-1,1,-1,1,2,1,2,2,-1,1,NaN
3,96620_201603,2437,2437,Residents need to be informed about what to do...,"Completing the integrated rescue system, inclu...",Improving state support for volunteer fire bri...,14,Residents need to be informed about what to do...,"EU, NATO and European structures",2,2,2,2,2,2,2,2,2,2,NaN
4,82430_201310,293,293,We will support the establishment of a special...,We will introduce a central register of accoun...,"We will introduce an electronic file, the so-c...",92,We will support the establishment of a special...,"EU, NATO and European structures",-1,1,-1,1,-1,1,1,1,-1,1,NaN


In [10]:
# manual annotations to numeric labels
df_coded["agreement_self_other"] = (
    pd.to_numeric(df_coded["agreement_self_other"], errors="coerce")
    .astype("Int64")
    .astype(int)
)

df_coded["agreement_valence"] = (
    pd.to_numeric(df_coded["agreement_valence"], errors="coerce")
    .astype("Int64")
    .astype(int)
)

In [11]:
# Check the manual label distributions
print("agreement_self_other:")
print(df_coded["agreement_self_other"].value_counts(dropna=False))

print("\nagreement_valence:")
print(df_coded["agreement_valence"].value_counts(dropna=False))

agreement_self_other:
agreement_self_other
-1    81
 1    69
 2    50
Name: count, dtype: int64

agreement_valence:
agreement_valence
 1    137
 2     45
-1     18
Name: count, dtype: int64


In [12]:
# Add manifesto-level variables
vars_file = f"{PROJECT_DIR}/Manifestos_variables_120526.csv"

df_vars = pd.read_csv(vars_file)


In [13]:
df_vars.head()

,manifesto_id,party,date,language,country_code,year,month,partyname,parfam
0,80220_199412,80220,199412,english,BGR,1994,12,Bulgarian Socialist Party,30.0
1,80410_199412,80410,199412,english,BGR,1994,12,Union of Democratic Forces,40.0
2,80811_199412,80811,199412,english,BGR,1994,12,People’s Union,70.0
3,80951_199412,80951,199412,english,BGR,1994,12,Movement for Rights and Freedoms,90.0
4,80221_200907,80221,200907,bulgarian,BGR,2009,7,Coalition for Bulgaria,30.0


In [14]:
df_coded_with_vars = df_coded.merge(
    df_vars,
    how="left",
    left_on="docname",
    right_on="manifesto_id"
)

In [15]:
df_coded_with_vars.head()

,docname,from,to,pre,keyword,post,lenght,text,referent_category,self/other_SP,...,comments,manifesto_id,party,date,language,country_code,year,month,partyname,parfam
0,97522_201407,1647,1647,National Parliaments must be more closely invo...,"NSi understands that national parliaments, tog...","In order to shape the future, it is essential ...",16,National Parliaments must be more closely invo...,"EU, NATO and European structures",2,...,NaN,97522_201407,97522,201407,slovenian,SVN,2014,7,New Slovenian Christian People’s Party,50.0
1,88440_201210,373,373,We will ensure Lithuania's security through ac...,We will effectively participate in the resolut...,Ensuring that Lithuanian representatives are a...,25,We will ensure Lithuania's security through ac...,"EU, NATO and European structures",1,...,NaN,88440_201210,88440,201210,lithuanian,LTU,2012,10,Labour Party,40.0
2,97330_200410,1750,1750,5.,Air transport modernisation of air traffic con...,6.,87,5. Air transport modernisation of air traffic ...,"EU, NATO and European structures",-1,...,NaN,97330_200410,97330,200410,slovenian,SVN,2004,10,Slovenian Democratic Party,60.0
3,96620_201603,2437,2437,Residents need to be informed about what to do...,"Completing the integrated rescue system, inclu...",Improving state support for volunteer fire bri...,14,Residents need to be informed about what to do...,"EU, NATO and European structures",2,...,NaN,96620_201603,96620,201603,slovak,SVK,2016,3,Ordinary People and Independent Personalities,60.0
4,82430_201310,293,293,We will support the establishment of a special...,We will introduce a central register of accoun...,"We will introduce an electronic file, the so-c...",92,We will support the establishment of a special...,"EU, NATO and European structures",-1,...,NaN,82430_201310,82430,201310,czech,CZE,2013,10,ANO 2011,40.0


In [16]:
country_map = {
    "BGR": "Bulgaria",
    "CZE": "Czech Republic",
    "EST": "Estonia",
    "HRV": "Croatia",
    "HUN": "Hungary",
    "LTU": "Lithuania",
    "LVA": "Latvia",
    "POL": "Poland",
    "ROU": "Romania",
    "SVK": "Slovakia",
    "SVN": "Slovenia"
}

df_coded_with_vars["country_name"] = df_coded_with_vars["country_code"].map(country_map)

df_coded_with_vars[["country_code", "country_name"]].drop_duplicates().sort_values("country_code")

,country_code,country_name
7,BGR,Bulgaria
4,CZE,Czech Republic
12,EST,Estonia
25,HUN,Hungary
1,LTU,Lithuania
6,POL,Poland
13,ROU,Romania
3,SVK,Slovakia
0,SVN,Slovenia


In [17]:
TEXT_COLUMN = "keyword"              # target sentence
PREVIOUS_COLUMN = "pre"              # sentence before
NEXT_COLUMN = "post"                 # sentence after
REFERENT_COLUMN = "referent_category" # actor/referent already coded
COUNTRY_COLUMN = "country_name"       # country name

In [25]:
import json
import time
from tqdm import tqdm
from openai import OpenAI
from google.colab import userdata
import hashlib

In [19]:
# API key
api_key = userdata.get("OPENAI_API_KEY")

if api_key is None:
    raise ValueError("OPENAI_API_KEY not found in Colab Secrets.")

openai_client = OpenAI(api_key=api_key)

In [20]:
MODELS_TO_TEST = [
    #"gpt-6-sol",
    "gpt-6.1-sol",
    "gpt-6-luna"
]

In [21]:
schema_self_other_valence = {
    "type": "object",
    "properties": {
        "self_other": {
            "type": "integer",
            "enum": [-1, 1, 2]
        },
        "valence": {
            "type": "integer",
            "enum": [-1, 1, 2]
        }
    },
    "required": ["self_other", "valence"],
    "additionalProperties": False
}

In [22]:
def build_prompt_self_other_valence(
    target_sentence,
    country=None,
    referent_category=None,
    previous_sentence=None,
    next_sentence=None
):

    return f"""
You are coding sentences from party manifestos in Central and Eastern Europe. Your task is to code only two variables: self_other and valence.

**Inputs:**
Country of manifesto: {country}
West-related actor/referent category to code: {referent_category}
Previous sentence: {previous_sentence}
TARGET SENTENCE: {target_sentence}
Next sentence: {next_sentence}

**Important unit rule:**
• Code only the TARGET SENTENCE.
• Use the previous and next sentences to better understand the context, clarify pronouns, implicit referents, incomplete phrases, or ambiguous terms.
• Code the provided West-related actor/referent, not the sentence in general.
• If a sentence contains multiple geopolitical actors, do not average them.
• Code the two variables from the perspective of the manifesto country.
• Code self_other and valence independently.

**Axis 1: self_other**
Code how the target sentence positions the party's country/national community in relation to the provided West-related actor/referent.

**Allowed values:**

1 = self

**Definition:** The sentence presents the country as Western, European, or as part of, belonging to, acting within or on behalf of the West/Europe/EU/NATO/Western countries.

**Clarification:** The sentence does not need to explicitly say “we are European” or “we are members.” If it suggests equal relations with Western countries or full membership in the West or Europe, it is self.

**Negative clarification:** Positive does not mean self. For instance, the West/Europe or Western countries used as a comparison or model, the classification would not be self, it would be other.

**Positive example:** "The change of regime has confirmed the historical place of the Hungarian nation in Europe: we are not a ferry country between East and West, but unquestionably the West, the European Union and an integral part of the transatlantic community."

**Negative example coded as other:** "Internationally, Slovenia must continue to consolidate its role as a sovereign and equal member of the European Union and the Atlantic Alliance."

-1 = other

**Definition:** The sentence presents the West/Europe/EU/NATO/Western countries as external, or suggests that their country is not fully Western/European or not yet integrated into the West/Europe/EU/NATO.

**Clarification:** If the country needs to catch up or gain influence to be more similar to Western/European actors, it is classified as other.

**Negative clarification:** For instance, taking a position on behalf of Western/European countries or organisations or discussing common policy issues from an equal position, it is classified as self, not as other.

**Positive example:** "Through greater involvement in the European integration process and the development of transatlantic ties, we will fundamentally strengthen the international position and security of the Czech Republic."

**Negative example coded as self:** "We actively support the development of the European Union's common foreign and security policy."

2 = not applicable

**Definition:** There is no broader identity or relational meaning toward the West/Europe/Western countries/EU/NATO.

**Clarification:** This includes simple descriptive, geographic, and biographic mentions. For instance, geographic locations would be classified as not applicable.

**Negative clarification:** If the mention of a Western actor is meaningful, for instance, suggesting a higher or lower status, identity or membership, then it is not classified as not applicable.

**Positive example:** "In order to maintain a living link with the motherland, a new institutional system will be established to provide administrative, social, cultural, legal and contact support to the Hungarian diaspora in the West and to expatriates."

**Negative example coded as self:** "We will do our utmost to ensure that Lithuania develops and maximises all possible foreign policy instruments in its relations with Germany, France, Italy, Spain, the UK, Scandinavia and the Benelux countries, in line with its own positions on issues of principle."

**Axis 2: valence**
Code the evaluative or relational orientation of the target sentence toward the provided West-related actor/referent.

**Allowed values:**

1 = positive

**Definition:** The West/Europe/Western countries/EU/NATO are framed as positive, desirable, beneficial, legitimate, protective, supportive, advanced, useful, worth supporting, worth defending, worth improving, or part of the solution.

**Clarification:** Negative aspects in the sentence do not automatically mean negative if the West-related actor/referent remains framed positively.

**Negative clarification:** If the West/Europe/Western countries/EU/NATO are presented as negative warning cases, the mention would be classified as negative.

**Positive example:** "Today, in the face of such "societal" problems, it is time to think about how we would like to be, now that we are part of Europe, the West."

**Negative example coded as negative:** "It appears that the parties, referring to Brussels for their every action or thesis, which have ruled so far, do not even respect the principles of their beloved EU."

-1 = negative

**Definition:** The West/Europe/Western countries/EU/NATO are framed as negative, harmful, threatening, unstable, unfair, subordinating, imposing, failed, problematic, or as a negative warning example.

**Clarification:** Do not infer negative valence from the general negativity of the sentence.

**Negative clarification:** If the sentence discusses a negative aspect in the West/Europe/Western countries/EU/NATO but the general position towards the actor is still positive, the classification is positive.

**Positive example:** "Its root cause is that Western civilisation, of which Lithuania is a part, has in its historical development broken away from its classical - ancient and Christian - roots."

**Negative example coded as positive:** "We are also lagging far behind compared to the tendencies of the Western states."

2 = not applicable

**Definition:** There is no evaluative position toward the West/Europe/Western countries/EU/NATO.

**Clarification:** This includes simple descriptive, geographic, and biographic mentions. For instance, geographic locations would be classified as not applicable.

**Negative clarification:** If EU/NATO/Western institutions are presented as sources of security, funding, rights, standards, democracy, solidarity, or other benefits, and an evaluative position is attached to them, they are not classified as not applicable.

**Positive example:** "ENEMIES OF FREEDOM AND OUR RESISTANCE After the criminal German and Soviet occupation, basic values were rejected by the Communists using force and mass repression."

**Negative example coded as positive:** "Promoting an active and targeted policy towards countries in Europe's immediate neighbourhood, particularly in North Africa and the Middle East, to enhance stability and prosperity in these countries, thereby reducing security risks."

Return only valid JSON in this exact format:
{{
  "self_other": 1,
  "valence": 1
}}
"""

In [27]:
def code_self_other_valence(
    target_sentence,
    country=None,
    referent_category=None,
    previous_sentence=None,
    next_sentence=None,
    model=None,
    max_retries=3
):

    if pd.isna(target_sentence) or str(target_sentence).strip() == "":
        return {
            "self_other": None,
            "valence": None,
            "error": "empty target sentence",
            "input_tokens": None,
            "output_tokens": None,
            "total_tokens": None
        }

    prompt = build_prompt_self_other_valence(
        target_sentence=str(target_sentence),
        country="" if pd.isna(country) else str(country),
        referent_category="" if pd.isna(referent_category) else str(referent_category),
        previous_sentence="" if pd.isna(previous_sentence) else str(previous_sentence),
        next_sentence="" if pd.isna(next_sentence) else str(next_sentence)
    )

    for attempt in range(max_retries):
        try:
            response = openai_client.responses.create(
                model=model,
                input=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                reasoning={
                    "effort": "medium"
                },
                max_output_tokens=2000,
                text={
                    "format": {
                        "type": "json_schema",
                        "name": "west_self_other_valence",
                        "strict": True,
                        "schema": schema_self_other_valence
                    }
                }
            )

            # Check response status
            if response.status != "completed":
                raise ValueError(
                    f"Response incomplete: {response.status}"
                )

            # Parse JSON
            result = json.loads(response.output_text)

            # Validate labels
            valid_labels = [-1, 1, 2]

            if (
                result.get("self_other") not in valid_labels
                or result.get("valence") not in valid_labels
            ):
                raise ValueError(f"Invalid labels: {result}")

            # Token usage
            usage = (
                response.usage.model_dump()
                if response.usage is not None
                else {}
            )

            return {
                "self_other": result["self_other"],
                "valence": result["valence"],
                "error": None,
                "input_tokens": usage.get("input_tokens"),
                "output_tokens": usage.get("output_tokens"),
                "total_tokens": usage.get("total_tokens")
            }

        except Exception as e:

            if attempt == max_retries - 1:
                return {
                    "self_other": None,
                    "valence": None,
                    "error": str(e),
                    "input_tokens": None,
                    "output_tokens": None,
                    "total_tokens": None
                }

            time.sleep(2 ** attempt)

In [24]:
# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    cohen_kappa_score
)


In [56]:

# GPT annotations

PROJECT_DIR = Path(PROJECT_DIR)

MODELS_TO_TEST = [
    "gpt-6.1-sol",
    "gpt-6-luna",
    "gpt-5.6-sol"
]

MODEL_FILES = {
    "gpt-6.1-sol": PROJECT_DIR / "west_self_other_valence_gpt-6_1-sol_20261009_medium_time_cost.csv",
    "gpt-6-luna": PROJECT_DIR / "west_self_other_valence_gpt-6-luna_20261009_medium_time_cost.csv",
    "gpt-5.6-sol": PROJECT_DIR / "west_self_other_valence_gpt-5_6-sol_20261011_medium_reproducibility.csv"
}

# USD per million tokens
PRICES = {
    "gpt-6.1-sol": {
        "input": 2.00,
        "cached_input": 0.10,
        "output": 10.00
    },
    "gpt-6-luna": {
        "input": 0.10,
        "cached_input": 0.01,
        "output": 0.50
    },
    "gpt-5.6-sol": {
        "input": 4.00,
        "cached_input": 0.40,
        "output": 20.00
    }
}

RUN_API = False
TEST_ONLY = False
CHECKPOINT_EVERY = 1
MAX_CONSECUTIVE_ERRORS = 3

LABELS = [-1, 1, 2]
all_model_results = []


def save_checkpoint(data, path):
    temp_path = path.with_suffix(".csv.tmp")
    data.to_csv(temp_path, index=False, encoding="utf-8")
    os.replace(temp_path, path)


for MODEL in MODELS_TO_TEST:

    print(f"\nMODEL: {MODEL}")

    OUTPUT_FILE = MODEL_FILES[MODEL]

    # Load saved annotations
    if OUTPUT_FILE.exists():

        df_model = pd.read_csv(OUTPUT_FILE)

        print(f"Loaded existing file: {OUTPUT_FILE}")
        print(f"Total sentences: {len(df_model)}")

        if {"self_other_api_new", "valence_api_new"}.issubset(df_model.columns):

            completed = (
                pd.to_numeric(
                    df_model["self_other_api_new"],
                    errors="coerce"
                ).isin(LABELS)
                &
                pd.to_numeric(
                    df_model["valence_api_new"],
                    errors="coerce"
                ).isin(LABELS)
            )

            print(f"Completed annotations: {int(completed.sum())}")
            print(f"Remaining annotations: {int((~completed).sum())}")

        all_model_results.append({
            "model": MODEL,
            "df": df_model,
            "session_seconds": 0
        })

        print("Skipping API annotations.")
        continue

    if not RUN_API:
        print(f"File not found: {OUTPUT_FILE}")
        print("API disabled. Skipping model.")
        continue

    if not OUTPUT_FILE.parent.is_dir():
        raise FileNotFoundError(
            f"Output folder not found: {OUTPUT_FILE.parent}"
        )

    model_start = time.perf_counter()

    df_model = df_coded_with_vars.reset_index(drop=True).copy()

    for col in [PREVIOUS_COLUMN, NEXT_COLUMN, COUNTRY_COLUMN]:
        if col not in df_model.columns:
            df_model[col] = ""

    required_columns = [
        TEXT_COLUMN,
        REFERENT_COLUMN,
        "agreement_self_other",
        "agreement_valence"
    ]

    for col in required_columns:
        if col not in df_model.columns:
            raise ValueError(f"Missing column: {col}")

    output_columns = [
        "self_other_api_new",
        "valence_api_new",
        "api_error",
        "input_tokens",
        "output_tokens",
        "total_tokens",
        "cached_input_tokens",
        "elapsed_seconds",
        "estimated_cost_usd"
    ]

    for col in output_columns:
        if col not in df_model.columns:
            df_model[col] = None

    df_model["model"] = MODEL

    rows_to_process = df_model.index.tolist()

    if TEST_ONLY:
        rows_to_process = rows_to_process[:1]

    print(f"Total sentences: {len(df_model)}")
    print(f"Rows to process: {len(rows_to_process)}")

    consecutive_errors = 0
    attempts = 0

    try:

        for i in tqdm(rows_to_process, desc=MODEL):

            target = df_model.loc[i, TEXT_COLUMN]

            if pd.isna(target) or str(target).strip() == "":
                df_model.loc[i, "api_error"] = "empty target"
                save_checkpoint(df_model, OUTPUT_FILE)
                continue

            request_start = time.perf_counter()

            try:
                result = code_self_other_valence(
                    target_sentence=target,
                    country=df_model.loc[i, COUNTRY_COLUMN],
                    referent_category=df_model.loc[i, REFERENT_COLUMN],
                    previous_sentence=df_model.loc[i, PREVIOUS_COLUMN],
                    next_sentence=df_model.loc[i, NEXT_COLUMN],
                    model=MODEL,
                    max_retries=1 if TEST_ONLY else 3
                )

            except Exception as e:
                result = {
                    "self_other": None,
                    "valence": None,
                    "error": f"{type(e).__name__}: {e}",
                    "input_tokens": None,
                    "output_tokens": None,
                    "total_tokens": None
                }

            elapsed = time.perf_counter() - request_start
            attempts += 1

            df_model.loc[i, "self_other_api_new"] = result.get("self_other")
            df_model.loc[i, "valence_api_new"] = result.get("valence")
            df_model.loc[i, "api_error"] = result.get("error")

            input_tokens = result.get("input_tokens")
            output_tokens = result.get("output_tokens")
            total_tokens = result.get("total_tokens")
            cached_tokens = result.get("cached_input_tokens")

            df_model.loc[i, "input_tokens"] = input_tokens
            df_model.loc[i, "output_tokens"] = output_tokens
            df_model.loc[i, "total_tokens"] = total_tokens
            df_model.loc[i, "cached_input_tokens"] = cached_tokens
            df_model.loc[i, "elapsed_seconds"] = elapsed

            # Estimate costs
            if input_tokens is not None and output_tokens is not None:

                cached = cached_tokens if cached_tokens is not None else 0
                uncached = max(input_tokens - cached, 0)

                estimated_cost = (
                    uncached * PRICES[MODEL]["input"]
                    + cached * PRICES[MODEL]["cached_input"]
                    + output_tokens * PRICES[MODEL]["output"]
                ) / 1_000_000

                df_model.loc[i, "estimated_cost_usd"] = estimated_cost

            self_value = pd.to_numeric(
                result.get("self_other"), errors="coerce"
            )
            valence_value = pd.to_numeric(
                result.get("valence"), errors="coerce"
            )

            success = (
                self_value in LABELS
                and valence_value in LABELS
            )

            if success:
                consecutive_errors = 0
                print(f"SUCCESS: {MODEL}, row {i}")

            else:
                consecutive_errors += 1
                print(f"ERROR: {MODEL}, row {i}: {result.get('error')}")

            if attempts % CHECKPOINT_EVERY == 0:
                save_checkpoint(df_model, OUTPUT_FILE)

            if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                print(f"Stopping {MODEL}: repeated errors.")
                break

    except Exception as e:
        print(f"Unexpected error: {type(e).__name__}: {e}")

    finally:
        save_checkpoint(df_model, OUTPUT_FILE)
        print(f"FILE SAVED: {OUTPUT_FILE}")

    model_seconds = time.perf_counter() - model_start

    completed = (
        pd.to_numeric(
            df_model["self_other_api_new"],
            errors="coerce"
        ).isin(LABELS)
        &
        pd.to_numeric(
            df_model["valence_api_new"],
            errors="coerce"
        ).isin(LABELS)
    )

    all_model_results.append({
        "model": MODEL,
        "df": df_model,
        "session_seconds": model_seconds
    })

    print(f"MODEL FINISHED: {MODEL}")
    print(f"API calls attempted: {attempts}")
    print(f"Successful annotations: {int(completed.sum())}/{len(df_model)}")
    print(f"Session time: {model_seconds / 60:.2f} minutes")



MODEL: gpt-6.1-sol
Loaded existing file: /content/drive/MyDrive/West_Narratives/west_self_other_valence_gpt-6_1-sol_20261009_medium_time_cost.csv
Total sentences: 200
Completed annotations: 200
Remaining annotations: 0
Skipping API annotations.

MODEL: gpt-6-luna
Loaded existing file: /content/drive/MyDrive/West_Narratives/west_self_other_valence_gpt-6-luna_20261009_medium_time_cost.csv
Total sentences: 200
Completed annotations: 200
Remaining annotations: 0
Skipping API annotations.

MODEL: gpt-5.6-sol
Loaded existing file: /content/drive/MyDrive/West_Narratives/west_self_other_valence_gpt-5_6-sol_20261011_medium_reproducibility.csv
Total sentences: 200
Completed annotations: 200
Remaining annotations: 0
Skipping API annotations.


Check saved GPT results

In [47]:

print("\nTEST RESULTS")

for model, file in MODEL_FILES.items():

    if not file.exists():
        print(f"\n{model}: NO FILE")
        continue

    data = pd.read_csv(file)

    self_values = pd.to_numeric(
        data["self_other_api_new"], errors="coerce"
    )

    valence_values = pd.to_numeric(
        data["valence_api_new"], errors="coerce"
    )

    completed = (
        self_values.isin([-1, 1, 2])
        & valence_values.isin([-1, 1, 2])
    )

    print(f"\nMODEL: {model}")
    print("Total sentences:", len(data))
    print("Successfully coded:", completed.sum())
    print("Remaining:", (~completed).sum())

    if "api_error" in data.columns:
        errors = data["api_error"].dropna()

        if len(errors):
            print("API errors:")
            print(errors.head(3).to_string(index=False))
        else:
            print("No API errors recorded.")



TEST RESULTS

MODEL: gpt-6.1-sol
Total sentences: 200
Successfully coded: 200
Remaining: 0
No API errors recorded.

MODEL: gpt-6-luna
Total sentences: 200
Successfully coded: 200
Remaining: 0
No API errors recorded.

MODEL: gpt-5.6-sol
Total sentences: 200
Successfully coded: 200
Remaining: 0
No API errors recorded.


Compare models

In [44]:

MODEL_FILES_SP = {
    "GPT-6 Sol": "west_self_other_valence_gpt-6-sol_20261009_medium.csv",
    "GPT-6.1 Sol": "west_self_other_valence_gpt-6_1-sol_20261009_medium_time_cost.csv",
    "GPT-6 Luna": "west_self_other_valence_gpt-6-luna_20261009_medium_time_cost.csv",
    "GPT-5.6 Sol": "west_self_other_valence_gpt-5_6-sol_20261011_medium_reproducibility.csv",
    "GPT-5.4": "west_self_other_valence_gpt-5_4_20261011_medium_reproducibility.csv"
}

LABELS = [-1, 1, 2]

performance_sp = []
reports_sp = {}
confusions_sp = {}
disagreements_sp = []

# Compare with SP annotations
for model, filename in MODEL_FILES_SP.items():

    file = PROJECT_DIR / filename

    if not file.exists():
        print(f"FILE NOT FOUND: {file}")
        continue

    data = pd.read_csv(file)

    for axis, manual_col, pred_col, class_names in [
        (
            "self_other",
            "self/other_SP",
            "self_other_api_new",
            ["Other", "Self", "Not applicable"]
        ),
        (
            "valence",
            "valence_SP",
            "valence_api_new",
            ["Negative", "Positive", "Not applicable"]
        )
    ]:

        if manual_col not in data.columns:
            print(f"{model}: missing {manual_col}")
            continue

        manual = pd.to_numeric(
            data[manual_col], errors="coerce"
        )

        predicted = pd.to_numeric(
            data[pred_col], errors="coerce"
        )

        valid_manual = manual.isin(LABELS)
        valid = valid_manual & predicted.isin(LABELS)

        n_manual = int(valid_manual.sum())
        n_scored = int(valid.sum())

        if n_scored == 0:
            print(f"No valid observations: {model}, {axis}")
            continue

        y_true = manual[valid].astype(int)
        y_pred = predicted[valid].astype(int)

        # Main metrics
        performance_sp.append({
            "model": model,
            "axis": axis,
            "n_manual": n_manual,
            "n_scored": n_scored,
            "coverage": n_scored / n_manual,
            "accuracy": accuracy_score(y_true, y_pred),
            "balanced_accuracy": balanced_accuracy_score(
                y_true, y_pred
            ),
            "macro_f1": f1_score(
                y_true, y_pred,
                labels=LABELS,
                average="macro",
                zero_division=0
            ),
            "weighted_f1": f1_score(
                y_true, y_pred,
                labels=LABELS,
                average="weighted",
                zero_division=0
            ),
            "cohens_kappa": cohen_kappa_score(
                y_true, y_pred,
                labels=LABELS
            )
        })

        # Classification report
        reports_sp[(model, axis)] = pd.DataFrame(
            classification_report(
                y_true,
                y_pred,
                labels=LABELS,
                target_names=class_names,
                output_dict=True,
                zero_division=0
            )
        ).T

        # Confusion matrix
        confusions_sp[(model, axis)] = pd.DataFrame(
            confusion_matrix(
                y_true, y_pred,
                labels=LABELS
            ),
            index=[f"SP {x}" for x in class_names],
            columns=[f"Model {x}" for x in class_names]
        )

        # Disagreements
        mismatches = data.loc[valid].copy()
        mismatches["model"] = model
        mismatches["axis"] = axis
        mismatches["SP_label"] = y_true.values
        mismatches["model_label"] = y_pred.values

        mismatches = mismatches[
            mismatches["SP_label"] != mismatches["model_label"]
        ]

        disagreements_sp.append(mismatches)

# Model comparison
model_comparison_sp = pd.DataFrame(performance_sp)

print("\nMODEL PERFORMANCE AGAINST SP")

display(
    model_comparison_sp.sort_values(
        ["axis", "macro_f1"],
        ascending=[True, False]
    ).round(4)
)

# Per-class scores
for (model, axis), report in reports_sp.items():
    print(f"\n{model} — {axis} (SP annotations)")
    display(report.round(3))

# Confusion matrices
for (model, axis), matrix in confusions_sp.items():
    print(f"\n{model} — {axis}")
    display(matrix)

# Coding disagreements
if disagreements_sp:

    disagreements_sp_df = pd.concat(
        disagreements_sp,
        ignore_index=True
    )

    print("\nDISAGREEMENTS WITH SP")

    display(
        disagreements_sp_df[
            [
                "model",
                "axis",
                TEXT_COLUMN,
                "SP_label",
                "model_label"
            ]
        ]
    )

# Save results
model_comparison_sp.to_csv(
    PROJECT_DIR / "five_models_performance_SP.csv",
    index=False
)

if disagreements_sp:
    disagreements_sp_df.to_csv(
        PROJECT_DIR / "five_models_disagreements_SP.csv",
        index=False
    )

print("\nSP evaluation complete. No API requests made.")


MODEL PERFORMANCE AGAINST SP


,model,axis,n_manual,n_scored,coverage,accuracy,balanced_accuracy,macro_f1,weighted_f1,cohens_kappa
2,GPT-6.1 Sol,self_other,200,200,1.0,0.750,0.7367,0.7376,0.7500,0.6076
0,GPT-6 Sol,self_other,200,200,1.0,0.740,0.6952,0.7060,0.7333,0.5833
6,GPT-5.6 Sol,self_other,200,200,1.0,0.715,0.6746,0.6835,0.7095,0.5447
4,GPT-6 Luna,self_other,200,200,1.0,0.680,0.6696,0.6658,0.6810,0.5009
8,GPT-5.4,self_other,200,200,1.0,0.675,0.6531,0.6558,0.6726,0.4856
5,GPT-6 Luna,valence,200,200,1.0,0.735,0.6902,0.6672,0.7425,0.4560
9,GPT-5.4,valence,200,200,1.0,0.755,0.6790,0.6501,0.7528,0.4762
3,GPT-6.1 Sol,valence,200,200,1.0,0.765,0.6666,0.6469,0.7597,0.4865
7,GPT-5.6 Sol,valence,200,200,1.0,0.775,0.6542,0.6407,0.7667,0.4979
1,GPT-6 Sol,valence,200,200,1.0,0.760,0.6435,0.6254,0.7439,0.4528



GPT-6 Sol — self_other (SP annotations)


,precision,recall,f1-score,support
Other,0.763,0.816,0.789,87.00
Self,0.716,0.795,0.753,73.00
Not applicable,0.731,0.475,0.576,40.00
accuracy,0.740,0.740,0.740,0.74
macro avg,0.737,0.695,0.706,200.00
weighted avg,0.740,0.740,0.733,200.00



GPT-6 Sol — valence (SP annotations)


,precision,recall,f1-score,support
Negative,0.545,0.706,0.615,17.00
Positive,0.840,0.906,0.872,139.00
Not applicable,0.500,0.318,0.389,44.00
accuracy,0.760,0.760,0.760,0.76
macro avg,0.628,0.644,0.625,200.00
weighted avg,0.740,0.760,0.744,200.00


... (35 outputs left out)

,Model Negative,Model Positive,Model Not applicable
SP Negative,13,2,2
SP Positive,0,120,19
SP Not applicable,10,16,18



DISAGREEMENTS WITH SP


,model,axis,keyword,SP_label,model_label
0,GPT-6 Sol,self_other,"NSi understands that national parliaments, tog...",2,1
1,GPT-6 Sol,self_other,"Completing the integrated rescue system, inclu...",2,1
2,GPT-6 Sol,self_other,Cohesion funding The successful implementation...,1,2
3,GPT-6 Sol,self_other,"In view of its political views and positions, ...",1,-1
4,GPT-6 Sol,self_other,We will defend the equal position of Czech far...,-1,1
...,...,...,...,...,...
525,GPT-5.4,valence,More than anywhere else in Europe.,1,2
526,GPT-5.4,valence,The construction of cross-border capacities in...,1,2
527,GPT-5.4,valence,"And they are getting worse, both in Europe and...",-1,2
528,GPT-5.4,valence,Our customs are notorious for their corruption...,1,2



SP evaluation complete. No API requests made.


Annotation with Antrhopic


In [52]:

# Claude models

!pip install anthropic -q
import anthropic
import json
import time
import os
import pandas as pd
from tqdm.auto import tqdm
from google.colab import userdata

RUN_API = False # change to True to run

MODELS = {
    "opus": {
        "id": "claude-opus-5-5",
        "in_per_m": 4.00,
        "out_per_m": 20.00
    },
    "sonnet": {
        "id": "claude-sonnet-5-5",
        "in_per_m": 2.00,
        "out_per_m": 10.00
    },
    "haiku": {
        "id": "claude-haiku-5-5",
        "in_per_m": 0.10,
        "out_per_m": 0.50
    }
}

OUTPUT_CSV = PROJECT_DIR / "west_self_other_valence_api_claude.csv"
SUMMARY_CSV = PROJECT_DIR / "west_self_other_valence_run_summary.csv"

MAX_TOKENS = 4096
MAX_RETRIES = 3
EFFORT = "medium"
SLEEP = 0.5

LABELS = [-1, 1, 2]

INPUT_DF = df_coded_with_vars.reset_index(drop=True).copy()

anthropic_client = None

def get_anthropic_client():
    global anthropic_client

    if anthropic_client is None:
        api_key = userdata.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise ValueError("ANTHROPIC_API_KEY not found in Colab Secrets")

        anthropic_client = anthropic.Anthropic(api_key=api_key)

    return anthropic_client


def save_claude(data):
    temp_file = OUTPUT_CSV.with_suffix(".csv.tmp")
    data.to_csv(temp_file, index=False)
    os.replace(temp_file, OUTPUT_CSV)


# Label one sentence
def label_row(row, model_id):

    prompt = build_prompt_self_other_valence(
        target_sentence=row.get(TEXT_COLUMN, ""),
        country=row.get(COUNTRY_COLUMN, ""),
        referent_category=row.get(REFERENT_COLUMN, ""),
        previous_sentence=row.get(PREVIOUS_COLUMN, ""),
        next_sentence=row.get(NEXT_COLUMN, "")
    )

    input_tokens = 0
    output_tokens = 0
    start = time.perf_counter()
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):

        try:
            output_config = {
                "format": {
                    "type": "json_schema",
                    "schema": schema_self_other_valence
                }
            }

            if EFFORT:
                output_config["effort"] = EFFORT

            response = get_anthropic_client().messages.create(
                model=model_id,
                max_tokens=MAX_TOKENS,
                messages=[{
                    "role": "user",
                    "content": prompt
                }],
                output_config=output_config
            )

            input_tokens += response.usage.input_tokens
            output_tokens += response.usage.output_tokens

            if response.stop_reason == "max_tokens":
                raise ValueError("Maximum output tokens reached")

            text = next(
                block.text
                for block in response.content
                if block.type == "text"
            )

            result = json.loads(text)

            if (
                result["self_other"] not in LABELS
                or result["valence"] not in LABELS
            ):
                raise ValueError("Invalid classification labels")

            return {
                "self_other": result["self_other"],
                "valence": result["valence"],
                "in_tok": input_tokens,
                "out_tok": output_tokens,
                "seconds": time.perf_counter() - start,
                "error": None
            }

        except anthropic.BadRequestError:
            raise

        except Exception as e:
            last_error = str(e)
            print(f"Attempt {attempt}: {last_error}")

            if attempt < MAX_RETRIES:
                time.sleep(2 ** attempt)

    return {
        "self_other": None,
        "valence": None,
        "in_tok": input_tokens,
        "out_tok": output_tokens,
        "seconds": time.perf_counter() - start,
        "error": last_error
    }


# Load previous results
if OUTPUT_CSV.exists():

    df_claude = pd.read_csv(OUTPUT_CSV).reset_index(drop=True)

    print("Loaded existing Claude results:", OUTPUT_CSV)

    check_columns = [
        TEXT_COLUMN,
        PREVIOUS_COLUMN,
        NEXT_COLUMN,
        REFERENT_COLUMN,
        COUNTRY_COLUMN
    ]

    if len(df_claude) != len(INPUT_DF):
        raise ValueError("Saved file and input data have different row counts")

    for col in check_columns:
        if col not in df_claude.columns or col not in INPUT_DF.columns:
            raise ValueError(f"Missing input column: {col}")

        saved = df_claude[col].fillna("").astype(str)
        original = INPUT_DF[col].fillna("").astype(str)

        if not saved.equals(original):
            raise ValueError(f"Input data mismatch in {col}")

else:

    df_claude = INPUT_DF.copy()
    print("No previous Claude results found")


# Prepare result columns
for model_name in MODELS:

    columns = [
        f"self_other_{model_name}",
        f"valence_{model_name}",
        f"in_tok_{model_name}",
        f"out_tok_{model_name}",
        f"sec_{model_name}",
        f"error_{model_name}"
    ]

    for col in columns:
        if col not in df_claude.columns:
            df_claude[col] = None


# Check saved annotations and run missing ones
for model_name, cfg in MODELS.items():

    print(f"\nMODEL: {cfg['id']}")

    self_col = f"self_other_{model_name}"
    valence_col = f"valence_{model_name}"

    completed = (
        pd.to_numeric(
            df_claude[self_col], errors="coerce"
        ).isin(LABELS)
        &
        pd.to_numeric(
            df_claude[valence_col], errors="coerce"
        ).isin(LABELS)
    )

    pending = df_claude.index[~completed].tolist()

    print("Total sentences:", len(df_claude))
    print("Already completed:", int(completed.sum()))
    print("Remaining:", len(pending))

    if not pending:
        print("All annotations already saved.")
        continue

    if not RUN_API:
        print("API disabled. Existing results preserved.")
        continue

    consecutive_errors = 0

    for i in tqdm(pending, desc=model_name):

        try:
            result = label_row(
                df_claude.loc[i],
                cfg["id"]
            )

            df_claude.at[i, self_col] = result["self_other"]
            df_claude.at[i, valence_col] = result["valence"]
            df_claude.at[i, f"in_tok_{model_name}"] = result["in_tok"]
            df_claude.at[i, f"out_tok_{model_name}"] = result["out_tok"]
            df_claude.at[i, f"sec_{model_name}"] = result["seconds"]
            df_claude.at[i, f"error_{model_name}"] = result["error"]

            if (
                result["self_other"] in LABELS
                and result["valence"] in LABELS
            ):
                consecutive_errors = 0
            else:
                consecutive_errors += 1

        except Exception as e:
            consecutive_errors += 1
            df_claude.at[i, f"error_{model_name}"] = str(e)
            print(f"Error in row {i}: {e}")

        finally:
            save_claude(df_claude)

        if consecutive_errors >= 3:
            print("Stopping after three consecutive errors.")
            break

        time.sleep(SLEEP)


# Save and summarize results
if RUN_API and not OUTPUT_CSV.exists():
    save_claude(df_claude)

summary = []

for model_name, cfg in MODELS.items():

    self_values = pd.to_numeric(
        df_claude[f"self_other_{model_name}"],
        errors="coerce"
    )

    valence_values = pd.to_numeric(
        df_claude[f"valence_{model_name}"],
        errors="coerce"
    )

    completed = (
        self_values.isin(LABELS)
        & valence_values.isin(LABELS)
    )

    input_tokens = pd.to_numeric(
        df_claude[f"in_tok_{model_name}"],
        errors="coerce"
    )

    output_tokens = pd.to_numeric(
        df_claude[f"out_tok_{model_name}"],
        errors="coerce"
    )

    seconds = pd.to_numeric(
        df_claude[f"sec_{model_name}"],
        errors="coerce"
    )

    total_input = input_tokens.sum(min_count=1)
    total_output = output_tokens.sum(min_count=1)

    if pd.notna(total_input) and pd.notna(total_output):
        cost = (
            total_input * cfg["in_per_m"]
            + total_output * cfg["out_per_m"]
        ) / 1_000_000
    else:
        cost = float("nan")

    recorded_seconds = seconds.sum(min_count=1)

    summary.append({
        "model": model_name,
        "model_id": cfg["id"],
        "rows_coded": len(df_claude),
        "rows_failed": int((~completed).sum()),
        "input_tokens": total_input,
        "output_tokens": total_output,
        "cost_usd": cost,
        "cost_per_1k_rows": (
            cost / len(df_claude) * 1000
            if len(df_claude) else float("nan")
        ),
        "api_time_s": recorded_seconds,
        "avg_s_per_row": seconds.mean()
    })

summary_df = pd.DataFrame(summary)

if OUTPUT_CSV.exists():
    summary_df.to_csv(SUMMARY_CSV, index=False)

print("\nCLAUDE RESULTS")
display(summary_df.round(4))

if OUTPUT_CSV.exists():
    print("Annotations:", OUTPUT_CSV)
    print("Summary:", SUMMARY_CSV)
else:
    print("No annotation file saved. Enable RUN_API to create one.")


No previous Claude results found

MODEL: claude-opus-5-5
Total sentences: 200
Already completed: 0
Remaining: 200


opus:   0%|          | 0/200 [00:00<?, ?it/s]

Error in row 181: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011ChKmqPgbZCYdTc8LsNKvB'}
Error in row 182: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011ChKmqSgfxkLKQDJDL7PRo'}
Error in row 183: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011ChKmqVY5rfFifFok9DyWY'}
Stopping after three consecutive errors.

MODEL: claude-sonnet-5-5
Total sentences: 200
Already completed: 0
Remaining: 200


sonnet:   0%|          | 0/200 [00:00<?, ?it/s]

Error in row 0: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011ChKmqWHimU9ZFguNuieqM'}
Error in row 1: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011ChKmqZ2Apjmef3SK6KPjv'}
Error in row 2: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011ChKmqbuosBS3S9DD4onEX'}
Stopping after three consecutive errors.

MODEL: claude-haiku-5-5
Total sentences: 200
Already completed: 0
Remaining: 200


haiku:   0%|          | 0/200 [00:00<?, ?it/s]

Error in row 0: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011ChKmqcehrqfsnJEe1vUmD'}
Error in row 1: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011ChKmqfVMtTqG2gLu2q4Ss'}
Error in row 2: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011ChKmqiVSvg1jCMsagchHb'}
Stopping after three consecutive errors.

CLAUDE RESULTS


,model,model_id,rows_coded,rows_failed,input_tokens,output_tokens,cost_usd,cost_per_1k_rows,api_time_s,avg_s_per_row
0,opus,claude-opus-5-5,200,19,480490.0,19715.0,2.3163,11.5813,405.5547,2.2406
1,sonnet,claude-sonnet-5-5,200,200,NaN,NaN,NaN,NaN,NaN,NaN
2,haiku,claude-haiku-5-5,200,200,NaN,NaN,NaN,NaN,NaN,NaN


Annotations: /content/drive/MyDrive/West_Narratives/west_self_other_valence_api_claude.csv
Summary: /content/drive/MyDrive/West_Narratives/west_self_other_valence_run_summary.csv


In [54]:

# Claude evaluation

df_claude = pd.read_csv(
    PROJECT_DIR / "west_self_other_valence_api_claude.csv"
)

results = []
reports = []
confusions = []

for model in ["opus", "sonnet", "haiku"]:
    for axis, manual_col in [
        ("self_other", "agreement_self_other"),
        ("valence", "agreement_valence")
    ]:

        manual = pd.to_numeric(df_claude[manual_col], errors="coerce")
        predicted = pd.to_numeric(df_claude[f"{axis}_{model}"], errors="coerce")

        valid_manual = manual.isin(LABELS)
        valid = valid_manual & predicted.isin(LABELS)

        if not valid.any():
            print(f"No valid results: {model}, {axis}")
            continue

        y_true = manual[valid].astype(int)
        y_pred = predicted[valid].astype(int)

        results.append({
            "model": model,
            "axis": axis,
            "n_scored": int(valid.sum()),
            "coverage": valid.sum() / valid_manual.sum(),
            "accuracy": accuracy_score(y_true, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(
                y_true, y_pred, labels=LABELS,
                average="macro", zero_division=0
            ),
            "weighted_f1": f1_score(
                y_true, y_pred, labels=LABELS,
                average="weighted", zero_division=0
            ),
            "cohens_kappa": cohen_kappa_score(y_true, y_pred, labels=LABELS)
        })

        report = classification_report(
            y_true, y_pred,
            labels=LABELS,
            output_dict=True,
            zero_division=0
        )

        for category in ["-1", "1", "2"]:
            reports.append({
                "model": model,
                "axis": axis,
                "category": category,
                **report[category]
            })

        cm = confusion_matrix(y_true, y_pred, labels=LABELS)

        for i, true_label in enumerate(LABELS):
            for j, pred_label in enumerate(LABELS):
                confusions.append({
                    "model": model,
                    "axis": axis,
                    "manual_label": true_label,
                    "predicted_label": pred_label,
                    "count": int(cm[i, j])
                })

metrics_df = pd.DataFrame(results)
reports_df = pd.DataFrame(reports)
confusion_df = pd.DataFrame(confusions)

display(metrics_df.round(4))

metrics_df.to_csv(PROJECT_DIR / "claude_model_performance.csv", index=False)
reports_df.to_csv(PROJECT_DIR / "claude_classification_reports.csv", index=False)
confusion_df.to_csv(PROJECT_DIR / "claude_confusion_matrices.csv", index=False)

print("Claude metrics saved.")


No valid results: sonnet, self_other
No valid results: sonnet, valence
No valid results: haiku, self_other
No valid results: haiku, valence


,model,axis,n_scored,coverage,accuracy,balanced_accuracy,macro_f1,weighted_f1,cohens_kappa
0,opus,self_other,181,0.905,0.6961,0.6584,0.6358,0.6590,0.5216
1,opus,valence,181,0.905,0.7514,0.6687,0.6278,0.7263,0.4559


Claude metrics saved.


Replicability for best perfoming models

In [ ]:
# GPT-6.1 SOL

MODEL = "gpt-6.1-sol"

RUN_TAG = "20261012_replication_2"

TEST_ONLY = False
RESUME_EXISTING = True
MAX_RETRIES = 1

LABELS = [-1, 1, 2]

OUTPUT_FILE = PROJECT_DIR / (
    f"west_self_other_valence_gpt-6_1-sol_{RUN_TAG}.csv"
)

df_run = df_coded_with_vars.reset_index(drop=True).copy()

for col in [PREVIOUS_COLUMN, NEXT_COLUMN, COUNTRY_COLUMN]:
    if col not in df_run.columns:
        df_run[col] = ""


if RESUME_EXISTING and OUTPUT_FILE.exists():

    df_model = pd.read_csv(OUTPUT_FILE)

    if len(df_model) != len(df_run):
        raise ValueError("Checkpoint size mismatch.")

    # Verify same inputs and ordering
    for col in [
        TEXT_COLUMN,
        REFERENT_COLUMN,
        COUNTRY_COLUMN,
        PREVIOUS_COLUMN,
        NEXT_COLUMN
    ]:
        if not df_model[col].fillna("").astype(str).equals(
            df_run[col].fillna("").astype(str)
        ):
            raise ValueError(f"Checkpoint mismatch: {col}")

    print("Resuming existing replication checkpoint.")

else:
    df_model = df_run.copy()

for col in [
    "self_other_api_new",
    "valence_api_new",
    "api_error",
    "input_tokens",
    "output_tokens",
    "total_tokens",
    "cached_input_tokens",
    "elapsed_seconds"
]:
    if col not in df_model.columns:
        df_model[col] = None

df_model["model"] = MODEL
df_model["replication"] = 2
df_model["reasoning_effort"] = "medium"


completed = (
    pd.to_numeric(
        df_model["self_other_api_new"],
        errors="coerce"
    ).isin(LABELS)
    &
    pd.to_numeric(
        df_model["valence_api_new"],
        errors="coerce"
    ).isin(LABELS)
)

pending = df_model.index[~completed].tolist()

if TEST_ONLY:
    pending = pending[:1]

print("Completed:", int(completed.sum()))
print("Pending:", int((~completed).sum()))
print("Requests planned:", len(pending))
print("Output:", OUTPUT_FILE)

consecutive_errors = 0

try:

    for i in tqdm(pending, desc="GPT-6.1 Sol Replication 2"):

        start = time.perf_counter()

        try:
            result = code_self_other_valence(
                target_sentence=df_model.loc[i, TEXT_COLUMN],
                country=df_model.loc[i, COUNTRY_COLUMN],
                referent_category=df_model.loc[i, REFERENT_COLUMN],
                previous_sentence=df_model.loc[i, PREVIOUS_COLUMN],
                next_sentence=df_model.loc[i, NEXT_COLUMN],
                model=MODEL,
                max_retries=MAX_RETRIES
            )

        except Exception as e:
            result = {
                "self_other": None,
                "valence": None,
                "error": f"{type(e).__name__}: {e}"
            }

        elapsed = time.perf_counter() - start

        df_model.loc[i, "self_other_api_new"] = result.get("self_other")
        df_model.loc[i, "valence_api_new"] = result.get("valence")
        df_model.loc[i, "api_error"] = result.get("error")
        df_model.loc[i, "elapsed_seconds"] = elapsed

        for col in [
            "input_tokens",
            "output_tokens",
            "total_tokens",
            "cached_input_tokens"
        ]:
            df_model.loc[i, col] = result.get(col)

        # Save after every annotation
        df_model.to_csv(OUTPUT_FILE, index=False)

        success = (
            result.get("self_other") in LABELS
            and result.get("valence") in LABELS
        )

        if success:
            consecutive_errors = 0
        else:
            consecutive_errors += 1
            print(f"Error in row {i}: {result.get('error')}")

        if consecutive_errors >= 3:
            print("Stopping after 3 consecutive errors.")
            break

finally:
    df_model.to_csv(OUTPUT_FILE, index=False)

completed_final = (
    pd.to_numeric(
        df_model["self_other_api_new"],
        errors="coerce"
    ).isin(LABELS)
    &
    pd.to_numeric(
        df_model["valence_api_new"],
        errors="coerce"
    ).isin(LABELS)
)

print("\nGPT-6.1 SOL — REPLICATION ROUND 2")
print("Successfully annotated:", int(completed_final.sum()))
print("Total sentences:", len(df_model))
print("Saved to:", OUTPUT_FILE)

Completed: 0
Pending: 200
Requests planned: 200
Output: /content/drive/MyDrive/West_Narratives/west_self_other_valence_gpt-6_1-sol_20261012_replication_2.csv


GPT-6.1 Sol Replication 2: 100%|██████████| 200/200 [11:26<00:00,  3.43s/it]


GPT-6.1 SOL — REPLICATION ROUND 2
Successfully annotated: 200
Total sentences: 200
Saved to: /content/drive/MyDrive/West_Narratives/west_self_other_valence_gpt-6_1-sol_20261012_replication_2.csv


In [55]:
# =====================================================
# REPLICATION COMPARISON
# =====================================================

original_file = PROJECT_DIR / (
    "west_self_other_valence_gpt-6_1-sol_"
    "20261009_medium_time_cost.csv"
)

replication_file = OUTPUT_FILE

original = pd.read_csv(original_file)
replication = pd.read_csv(replication_file)

if len(original) != len(replication):
    raise ValueError("The two runs contain different row counts.")

if not original[TEXT_COLUMN].fillna("").astype(str).equals(
    replication[TEXT_COLUMN].fillna("").astype(str)
):
    raise ValueError("Sentence ordering differs between runs.")

replication_metrics = []

for axis, column in [
    ("self_other", "self_other_api_new"),
    ("valence", "valence_api_new")
]:

    first = pd.to_numeric(original[column], errors="coerce")
    second = pd.to_numeric(replication[column], errors="coerce")

    valid = first.isin(LABELS) & second.isin(LABELS)

    y1 = first[valid].astype(int)
    y2 = second[valid].astype(int)

    replication_metrics.append({
        "axis": axis,
        "n_compared": int(valid.sum()),
        "agreement": accuracy_score(y1, y2),
        "cohens_kappa": cohen_kappa_score(
            y1, y2, labels=LABELS
        ),
        "macro_f1": f1_score(
            y1, y2, labels=LABELS,
            average="macro",
            zero_division=0
        ),
        "n_disagreements": int((y1 != y2).sum())
    })

replication_metrics = pd.DataFrame(replication_metrics)

print("\nGPT-6.1 SOL — REPLICATION STABILITY")

display(replication_metrics.round(4))

replication_metrics.to_csv(
    PROJECT_DIR / "gpt6_1_sol_replication_stability.csv",
    index=False
)


GPT-6.1 SOL — REPLICATION STABILITY


,axis,n_compared,agreement,cohens_kappa,macro_f1,n_disagreements
0,self_other,200,0.875,0.8000,0.8444,25
1,valence,200,0.895,0.7618,0.8508,21


In [ ]:
#
# CLAUDE SONNET 5.5

MODEL_NAME = "sonnet"
MODEL_ID = "claude-sonnet-5-5"

TEST_ONLY = False
LABELS = [-1, 1, 2]

# Use a separate file to preserve the original results
ORIGINAL_FILE = Path(OUTPUT_CSV)

REPLICATION_FILE = ORIGINAL_FILE.with_name(
    "west_self_other_valence_claude_sonnet_replication2.csv"
)

# Verify Anthropic client
if anthropic_client is None:
    anthropic_client = anthropic.Anthropic(
        api_key=userdata.get("ANTHROPIC_API_KEY")
    )

# =====================================================
# CHECK ORIGINAL INPUT
# =====================================================

original = pd.read_csv(ORIGINAL_FILE).reset_index(drop=True)
df_input = INPUT_DF.reset_index(drop=True).copy()

check_columns = [
    TEXT_COLUMN,
    PREVIOUS_COLUMN,
    NEXT_COLUMN,
    REFERENT_COLUMN,
    COUNTRY_COLUMN
]

if len(original) != len(df_input):
    raise ValueError("Original and replication samples differ in size.")

for col in check_columns:
    if not original[col].fillna("").astype(str).equals(
        df_input[col].fillna("").astype(str)
    ):
        raise ValueError(f"Input mismatch: {col}")

print("Original sample verified:", len(df_input), "sentences")

# =====================================================
# LOAD CHECKPOINT OR CREATE REPLICATION
# =====================================================

if REPLICATION_FILE.exists():

    df_rep = pd.read_csv(REPLICATION_FILE).reset_index(drop=True)

    if len(df_rep) != len(df_input):
        raise ValueError("Replication checkpoint size mismatch.")

    for col in check_columns:
        if not df_rep[col].fillna("").astype(str).equals(
            df_input[col].fillna("").astype(str)
        ):
            raise ValueError(f"Checkpoint mismatch: {col}")

    print("Resuming replication checkpoint.")

else:

    df_rep = df_input.copy()
    print("Starting new Sonnet replication.")

# =====================================================
# OUTPUT COLUMNS
# =====================================================

for col in [
    "self_other_sonnet",
    "valence_sonnet",
    "in_tok_sonnet",
    "out_tok_sonnet",
    "sec_sonnet",
    "error_sonnet"
]:
    if col not in df_rep.columns:
        df_rep[col] = None

# =====================================================
# IDENTIFY PENDING SENTENCES
# =====================================================

completed = (
    pd.to_numeric(
        df_rep["self_other_sonnet"], errors="coerce"
    ).isin(LABELS)
    &
    pd.to_numeric(
        df_rep["valence_sonnet"], errors="coerce"
    ).isin(LABELS)
)

pending = df_rep.index[~completed].tolist()

if TEST_ONLY:
    pending = pending[:1]

print("Completed:", int(completed.sum()))
print("Remaining:", int((~completed).sum()))
print("Requests planned:", len(pending))

# =====================================================
# ANNOTATE ONLY PENDING SENTENCES
# =====================================================

consecutive_errors = 0

for i in tqdm(pending, desc="Sonnet 5.5 Replication"):

    try:

        result = label_row(
            df_rep.loc[i],
            MODEL_ID
        )

        for col, key in [
            ("self_other_sonnet", "self_other"),
            ("valence_sonnet", "valence"),
            ("in_tok_sonnet", "in_tok"),
            ("out_tok_sonnet", "out_tok"),
            ("sec_sonnet", "seconds")
        ]:
            df_rep.loc[i, col] = result.get(key)

        success = (
            result.get("self_other") in LABELS
            and result.get("valence") in LABELS
        )

        if success:
            consecutive_errors = 0
            df_rep.loc[i, "error_sonnet"] = None
        else:
            consecutive_errors += 1
            df_rep.loc[i, "error_sonnet"] = "Invalid or missing labels"
            print(f"Failed row {i}")

    except Exception as e:

        consecutive_errors += 1
        df_rep.loc[i, "error_sonnet"] = str(e)
        print(f"Error in row {i}: {e}")

    finally:

        # Save immediately after each annotation
        df_rep.to_csv(REPLICATION_FILE, index=False)

    if consecutive_errors >= 3:
        print("Stopping after 3 consecutive failures.")
        break

    time.sleep(SLEEP)

# =====================================================
# FINAL STATUS
# =====================================================

completed_final = (
    pd.to_numeric(
        df_rep["self_other_sonnet"], errors="coerce"
    ).isin(LABELS)
    &
    pd.to_numeric(
        df_rep["valence_sonnet"], errors="coerce"
    ).isin(LABELS)
)

print("\nSONNET 5.5 REPLICATION")
print("Completed:", int(completed_final.sum()))
print("Total:", len(df_rep))
print("Saved:", REPLICATION_FILE)

Original sample verified: 200 sentences
Resuming replication checkpoint.
Completed: 1
Remaining: 199
Requests planned: 199


Sonnet 5.5 Replication: 100%|██████████| 199/199 [06:19<00:00,  1.90s/it]


SONNET 5.5 REPLICATION
Completed: 200
Total: 200
Saved: west_self_other_valence_claude_sonnet_replication2.csv


In [ ]:
# =====================================================
# SONNET 5.5 — REPLICATION STABILITY
# =====================================================

round1 = pd.read_csv(ORIGINAL_FILE)
round2 = pd.read_csv(REPLICATION_FILE)

for col in check_columns:
    if not round1[col].fillna("").astype(str).equals(
        round2[col].fillna("").astype(str)
    ):
        raise ValueError(f"Input mismatch: {col}")

replication_results = []

for axis in ["self_other", "valence"]:

    col = f"{axis}_sonnet"

    first = pd.to_numeric(round1[col], errors="coerce")
    second = pd.to_numeric(round2[col], errors="coerce")

    valid = first.isin(LABELS) & second.isin(LABELS)

    if not valid.any():
        print(f"No valid pairs for {axis}")
        continue

    y1 = first[valid].astype(int)
    y2 = second[valid].astype(int)

    replication_results.append({
        "model": "Claude Sonnet 5.5",
        "axis": axis,
        "n_compared": int(valid.sum()),
        "agreement": accuracy_score(y1, y2),
        "cohens_kappa": cohen_kappa_score(
            y1, y2, labels=LABELS
        ),
        "macro_f1": f1_score(
            y1, y2,
            labels=LABELS,
            average="macro",
            zero_division=0
        ),
        "n_disagreements": int((y1 != y2).sum())
    })

sonnet_replication_metrics = pd.DataFrame(replication_results)

print("\nCLAUDE SONNET 5.5 — REPLICATION STABILITY")

display(sonnet_replication_metrics.round(4))

sonnet_replication_metrics.to_csv(
    REPLICATION_FILE.with_name(
        "claude_sonnet_replication_stability.csv"
    ),
    index=False
)


CLAUDE SONNET 5.5 — REPLICATION STABILITY


,model,axis,n_compared,agreement,cohens_kappa,macro_f1,n_disagreements
0,Claude Sonnet 5.5,self_other,200,0.870,0.7911,0.8466,26
1,Claude Sonnet 5.5,valence,200,0.895,0.7818,0.8781,21
